# Variante `montage_heatmap` — Universal EEG Transformer


El **Universal EEG Transformer** en modo **montaje** unifica grabaciones con
**cualquier configuración de electrodos** hacia un espacio canónico y,
desde ahí, hacia las cuatro referencias de EEG:
`unipolar`, `bipolar`, `CAR` y `REST` (todas de `C = 64` canales).

### Idea: proyectar y refinar

1. Un montaje de `C_s` electrodos (p. ej. 10-20, `C_s = 19`) observa la
   actividad del cuero cabelludo solo en sus posiciones. La grabación real de
   esos electrodos se **simula** de forma fiel: se parte de la referencia al
   infinito (REST) del montaje completo y se observan sus columnas en los
   electrodos del montaje fuente; sobre esa observación se computan las 4
   referencias **del propio montaje** (sus operadores, con su lead field).

2. Una matriz de proyección **fija** `P (C_s × C)` lleva la observación al
   espacio canónico: `x ∈ R^{C_s} ↦ x·P ∈ R^C`.

3. El **autoencoder lineal universal** aprende a refinar la proyección y a
   estimar las 4 referencias canónicas desde `x·P`: la conversión
   `s→d` queda como la matriz `A_{s→d} = P·W^{enc}_s·W^{dec}_d` (C_s×C).

### Los dos mecanismos de proyección (`mapping.method`)

| Método | Variante | Mecanismo | Suavizado |
|---|---|---|---|
| `leadfield` | `montage_leadfield` | **Solución inversa**: estima los potenciales canónicos resolviendo `P = W_s (W_s G_s)^{+T} G_c^T` con el lead field analítico multicapa | SVD truncado a `C_s//3` (el problema está mal condicionado, cond ~1e15) |
| `spline` | `montage_heatmap` | **Heatmap/topomapa**: interpolación esférica (Perrin) de la actividad entre los electrodos, que se percibe como **manchas** sobre el cuero cabelludo | Ridge que crece con la `densidad`: a montajes más dispersos, más suavizado |

### Calidad de la proyección pura (sin aprendizaje)

Reconstrucción *round-trip* al canónico con **solo** la proyección
(`data/mapping_results.csv`, test, 12 sujetos):

| Montaje | leadfield (ve) | spline (ve) | nearest (ve) |
|---|---|---|---|
| 10-20 (19) | **0.273** | 0.104 | 0.099 |
| 10-10 (39) | **0.259** | 0.118 | 0.526 |

El modelo universal parte de esta línea base y aprende a superarla.

### Identidad de la variante

**`montage_heatmap`** — estimación por interpólation esférica (Perrin) — actividad en manchas (config `config/montage_heatmap.yaml`).


La variante **`montage_heatmap`** proyecta con **splines esféricos** (Perrin,
1989), la misma interpolación de los topomapas clásicos: el campo de
potenciales se extiende de forma suave entre los electrodos y se representa
como un **heatmap** sobre el cuero cabelludo donde cada foco de actividad
aparece como una **mancha**.

El suavizado es **sensible a la densidad** del montaje (`adaptive_smoothness`):
cuanto más dispersos los electrodos (menos canales fuente por zona), mayor la
regularización ridge del spline (`smoothness = base · C_c/C_s`). Así la misma
pipeline sirve para configuraciones densas (64) o muy sparse (8).

**Resultado esperado**: la proyección es más suave que la inversa (vuelve peor
en *round-trip* solo con `P`, ve ≈ 0.10 en 10-20) pero muy estable; el
autoencoder universal la refina hasta aproximar las referencias canónicas.

**Resultado obtenido** (12 sujetos, test): a pesar de partir de la peor
proyección pura (ve ≈ 0.05), el modelo la supera con holgura y es **la mejor
de las dos variantes de montaje** — RMSE cross ≈ **16.3 µV**, `r ≈ 0.746`,
`ve ≈ 0.58`. Las características suaves del spline son más fáciles de refinar
que los modos de la solución inversa mal condicionada.


### Metodología del experimento

1. **Dataset real** `eegbci` (PhysioNet), sujetos 1–12, corridas 1–2:
   228 269 muestras × 64 canales, filtrado bandpass 1–45 Hz, split por
   bloques (train/val/test). El montaje 10-20 es un subconjunto exacto de los
   64 canales.
2. **Proyección fija** `P` construida con `mapping.build_projection`
   (`leadfield` con truncado SVD automático `C_s//3`, `spline` con
   `smoothness = 1e-5 · C_c/C_s`).
3. **Inicialización lineal empírica**: `init_from_data` ajusta las matrices
   sobre las observaciones **proyectadas** (`x·P`), de modo que el latente
   (`C=64`) ancla al unipolar canónico y cada ruta arranca bien condicionada.
4. **Pérdida**: MSE estandarizado (Z-score por lote) sobre las 16 rutas
   (origen del montaje fuente → destino canónico).
5. **Evaluación (test)**: RMSE/correlación por ruta de las estimaciones del
   modelo contra las referencias canónicas verdaderas, comparadas con la
   **proyección analítica pura** `obs @ P` (columnas `*_proy`).
6. **Figuras**: curvas de aprendizaje, heatmap de RMSE por ruta y **mapas de
   calor del cuero cabelludo** (electrodos → manchas) en 4 etapas:
   observación → proyección analítica → modelo → verdad canónica.


> **Nota de reproducción:** el modelo ya entrenado (12 sujetos) está en `runs/montage_heatmap`. Con `FORCE = False` el notebook **reutiliza el checkpoint** sin reentrenar (segundos); póngalo en `True` solo para reentrenar desde cero.

## 1. Carga del experimento

Configuración YAML, dataset real cacheado y los **insumos de montaje** (observaciones del montaje fuente + proyección `P`).

In [ ]:
# ---- Configuración del entorno ----------------------------------------
# Renderizado inline de figuras (debe activarse antes de importar matplotlib)
%matplotlib inline

import os, sys
from pathlib import Path

# Ruta raíz del repo y paquetes propios (src/)
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Las rutas relativas (data/, runs/, config/) se resuelven contra la raíz,
# no contra el cwd del kernel (correcto incluso desde otro directorio).
os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from eeg_transform.nb import (
    config_table, evaluate_montage, load_experiment, montage_inputs,
    plot_routes, plot_scalp, plot_training, train_variant,
)
from eeg_transform.training.trainer import build_model

# Modo notebook: semillas fijas
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)
plt.rcParams["figure.dpi"] = 110

CONFIG = ROOT / "config/montage_heatmap.yaml"
FORCE  = False          # True = reentrenar desde cero ignorando el checkpoint

In [ ]:
cfg, ds = load_experiment(CONFIG)
mi = montage_inputs(cfg, ds)
print(ds.summary())
print(f"Montaje fuente: {mi.montage} ({len(mi.src_names)} electrodos) "
      f"via '{mi.method}' → canónico {ds.n_channels} canales")
config_table(cfg).set_index(["sección", "parámetro"])

## 2. Arquitectura

Autoencoder universal + **proyección fija** `P (C_s→C)`: la ruta efectiva `s→d` es `A = P·W_enc·W_dec` y opera sobre las observaciones del montaje fuente.

In [ ]:
# Arquitectura efectiva: autoencoder universal + proyección fija P
model = build_model(cfg, ds.n_channels, projection=mi.projection)
model.ensure_built()

n_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
print(f"Variante: {cfg.model.variant}  |  latente: {cfg.model.latent_dim}  |  "
      f"montaje origen: {len(mi.src_names)} → canónico: {ds.n_channels}  |  "
      f"parámetros entrenables: {n_params:,}")
print(f"Proyección P: {mi.projection.shape} "
      "(fija; el autoencoder refina y convierte de referencias)")
print(f"Ruta efectiva s→d: A = P · W_enc_s · W_dec_d")
_ = model  # se reutiliza en train/eval

## 3. Entrenamiento

Igual que las variantes canónicas (Adam, Z-score por lote, early stopping) pero sobre observaciones de montaje. Reutiliza el checkpoint si existe.

In [ ]:
# Entrenamiento (reutiliza best.weights.h5 si existe, salvo FORCE=True)
model, history = train_variant(cfg, ds, force=FORCE)
run_dir = Path(cfg.training.run_dir)
print(f"run_dir: {run_dir}")

## 4. Evaluación en test

Métricas del **modelo** vs la **proyección analítica pura** (`obs @ P`, columnas `*_proy`): la ganancia cuantifica el refinamiento que aprende el transformador.

In [ ]:
# Evaluación en test: RMSE/ve por ruta vs proyección analítica pura (P)
metrics_df, summary = evaluate_montage(cfg, ds, model, montage=mi)

print("=== RESUMEN MONTAJE (RMSE en µV): modelo vs proyección analítica ===")
print(summary.round(3).to_string(index=False))
print("\n=== DETALLE POR RUTA (test) — columnas *_proy = solo proyección P ===")
print(metrics_df.round(9).to_string(index=False))

## 5. Figuras inline

Curvas de aprendizaje, heatmap de RMSE por ruta (modelo) y los **mapas de calor del cuero cabelludo** que muestran la actividad como manchas desde el montaje fuente hasta la verdad canónica.

In [ ]:
# Curvas de aprendizaje (pérdida estandarizada y MSE real)
plot_training(run_dir / "history.csv")
plt.show()

In [ ]:
# Heatmap de RMSE real por ruta origen→destino (µV, escala log10)
plot_routes(metrics_df, value_col="rmse",
            title=f"RMSE real por ruta (µV) — {cfg.model.variant}")
plt.show()

In [ ]:
# Mapas de calor del cuero cabelludo: electrodos como manchas de actividad
# Filas = referencia; columnas: observación del montaje fuente → proyección
# analítica → modelo → verdad canónica (el instante de máxima amplitud).
plot_scalp(cfg, ds, model, montage=mi)
plt.show()

## Conclusiones

Consulte `docs/mapping_wip.md` y `docs/results_comparison.md` para la interpretación comparativa de la unificación de montajes frente a las variantes canónicas.